# Vesuvius V4 Inference

## Model Performance
- **Best Val Dice**: 0.6027 at epoch 150
- **Architecture**: ResEncUNet3D with SafeInstanceNorm3d
- **Patch Size**: 128×128×128
- **Training**: 600 epochs, Dice+CE+Skeleton loss

## Post-Processing Options
1. **Simple Threshold**: 0.5 (standard)
2. **Hysteresis Thresholding**: Two-threshold approach for better connectivity
3. **Morphological Closing**: Connect fragmented vessels

In [ ]:
!pip install imagecodecs connected-components-3d tifffile -q

In [ ]:
import os
import sys
import gc
import zipfile
import warnings
from pathlib import Path
from typing import Tuple, List

import numpy as np
import pandas as pd
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from scipy import ndimage as ndi
from skimage.morphology import remove_small_objects, ball, closing

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False
    print("cc3d not found, using scipy for connected components")

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    TEST_IMAGES_DIR = Path("/kaggle/input/vesuvius-challenge-surface-detection/test_images")
    TEST_CSV = Path("/kaggle/input/vesuvius-challenge-surface-detection/test.csv")
    
    # ==========================================================================
    # V4 MODEL CHECKPOINT - UPDATE THIS PATH
    # ==========================================================================
    CHECKPOINT_PATH = Path("/kaggle/input/v4-model/pytorch/default/1/checkpoints_v4/best_model.pth")
    
    OUTPUT_DIR = Path("/kaggle/working/submission_masks")
    SUBMISSION_ZIP = Path("/kaggle/working/submission.zip")
    
    # Architecture - MUST match V4 training
    PATCH_SIZE: Tuple[int, int, int] = (128, 128, 128)
    FEATURES: List[int] = [32, 64, 128, 256, 320, 320]
    N_BLOCKS: List[int] = [1, 3, 4, 6, 6, 6]
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # Inference settings
    OVERLAP: float = 0.5
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    
    # ==========================================================================
    # POST-PROCESSING OPTIONS
    # ==========================================================================
    # V4 model has higher Val Dice (0.6027) so outputs are better calibrated
    # Try simple threshold first, hysteresis if needed
    
    # Option 1: Simple threshold (recommended first)
    USE_SIMPLE_THRESHOLD: bool = True
    SIMPLE_THRESHOLD: float = 0.5
    
    # Option 2: Hysteresis thresholding (if simple gives poor connectivity)
    USE_HYSTERESIS: bool = False
    THRESHOLD_HIGH: float = 0.6  # Seeds (high confidence)
    THRESHOLD_LOW: float = 0.3   # Expansion boundary
    
    # Morphological closing (helps connect fragmented vessels)
    USE_CLOSING: bool = False
    CLOSING_RADIUS: int = 1
    
    # Dust removal
    MIN_COMPONENT_SIZE: int = 50

cfg = Config()
cfg.OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"Device: {cfg.DEVICE}")
if cfg.DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} (CUDA {props.major}.{props.minor})")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

print(f"\nPost-processing mode:")
if cfg.USE_SIMPLE_THRESHOLD:
    print(f"  Simple threshold: {cfg.SIMPLE_THRESHOLD}")
elif cfg.USE_HYSTERESIS:
    print(f"  Hysteresis: HIGH={cfg.THRESHOLD_HIGH}, LOW={cfg.THRESHOLD_LOW}")
print(f"  Closing: {cfg.USE_CLOSING} (radius={cfg.CLOSING_RADIUS})")
print(f"  Min component: {cfg.MIN_COMPONENT_SIZE}")

In [ ]:
# =============================================================================
# MODEL - SafeInstanceNorm3d (MUST match V4 training)
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    """InstanceNorm3d with safeguards against division by very small variances."""
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super(SafeInstanceNorm3d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        
        if self.affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)
    
    def forward(self, x):
        mean = x.mean(dim=[2, 3, 4], keepdim=True)
        var = x.var(dim=[2, 3, 4], keepdim=True, unbiased=False)
        var_safe = torch.clamp(var, min=self.eps)
        x_norm = (x - mean) / torch.sqrt(var_safe + self.eps)
        
        if self.affine:
            x_norm = self.weight.view(1, -1, 1, 1, 1) * x_norm + self.bias.view(1, -1, 1, 1, 1)
        
        return x_norm


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=True),
            SafeInstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )

    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)

    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    """V4 ResEncUNet with SafeInstanceNorm3d."""

    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
    ):
        super().__init__()

        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]

        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)

        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()

        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            )
            self.encoders.append(encoder)

            if use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(nn.Identity())

            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2, bias=True))

        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()

        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2, bias=True))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))

        self.final = nn.Conv3d(features[0], out_ch, 1, bias=True)

        # Deep supervision heads
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(4, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1, bias=True))

    def forward(self, x):
        enc_features = []

        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)

        enc_features = enc_features[::-1]
        x = enc_features[0]

        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)

        return self.final(x)

print("Model architecture defined")

In [ ]:
# =============================================================================
# POST-PROCESSING FUNCTIONS
# =============================================================================

def hysteresis_threshold_3d(prob_map, threshold_high, threshold_low):
    """Apply seeded hysteresis thresholding to a 3D probability map."""
    seeds = prob_map >= threshold_high
    weak = prob_map >= threshold_low
    struct = ndi.generate_binary_structure(3, 3)  # 26-connectivity
    result = ndi.binary_propagation(seeds, structure=struct, mask=weak)
    return result.astype(bool)


def apply_morphological_closing(mask, radius):
    """Apply morphological closing to connect nearby structures."""
    if radius <= 0:
        return mask
    selem = ball(radius)
    closed = closing(mask, selem)
    return closed


def remove_small_components(mask, min_size):
    """Remove small connected components (dust removal)."""
    if min_size <= 0:
        return mask
    cleaned = remove_small_objects(mask.astype(bool), min_size=min_size, connectivity=3)
    return cleaned


def postprocess(prob_map, cfg):
    """Full post-processing pipeline."""
    print(f"  Prob range: [{prob_map.min():.4f}, {prob_map.max():.4f}], mean: {prob_map.mean():.4f}")
    
    # Step 1: Thresholding
    if cfg.USE_SIMPLE_THRESHOLD:
        mask = prob_map >= cfg.SIMPLE_THRESHOLD
        thresh_pct = 100 * mask.mean()
        print(f"  Simple threshold ({cfg.SIMPLE_THRESHOLD}): {thresh_pct:.3f}% FG")
    elif cfg.USE_HYSTERESIS:
        mask = hysteresis_threshold_3d(
            prob_map,
            threshold_high=cfg.THRESHOLD_HIGH,
            threshold_low=cfg.THRESHOLD_LOW
        )
        thresh_pct = 100 * mask.mean()
        print(f"  Hysteresis (H={cfg.THRESHOLD_HIGH}, L={cfg.THRESHOLD_LOW}): {thresh_pct:.3f}% FG")
    else:
        mask = prob_map >= 0.5
        print(f"  Default threshold (0.5): {100 * mask.mean():.3f}% FG")
    
    # Step 2: Morphological closing (optional)
    if cfg.USE_CLOSING and cfg.CLOSING_RADIUS > 0:
        mask = apply_morphological_closing(mask, cfg.CLOSING_RADIUS)
        print(f"  After closing: {100 * mask.mean():.3f}% FG")
    
    # Step 3: Dust removal
    if cfg.MIN_COMPONENT_SIZE > 0:
        mask = remove_small_components(mask, cfg.MIN_COMPONENT_SIZE)
        print(f"  After dust removal: {100 * mask.mean():.3f}% FG")
    
    return mask.astype(np.uint8)

print("Post-processing functions ready")

In [ ]:
# =============================================================================
# NORMALIZATION & INFERENCE
# =============================================================================

def normalize_volume(volume: np.ndarray) -> np.ndarray:
    """Simple z-score normalization (matches V4 training)."""
    return (volume.astype(np.float32) - volume.mean()) / (volume.std() + 1e-8)


@torch.no_grad()
def sliding_window_inference(model, volume, patch_size=(128, 128, 128), overlap=0.5, device="cuda", use_amp=True):
    """Sliding window inference with Gaussian blending."""
    model.eval()
    model.to(device)
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd*(1-overlap)), int(ph*(1-overlap)), int(pw*(1-overlap))
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    # Gaussian blending weights
    gz = np.exp(-0.5*((np.arange(pd)-pd/2)/(pd*0.125))**2)
    gy = np.exp(-0.5*((np.arange(ph)-ph/2)/(ph*0.125))**2)
    gx = np.exp(-0.5*((np.arange(pw)-pw/2)/(pw*0.125))**2)
    gauss = (gz[:,None,None] * gy[None,:,None] * gx[None,None,:]).astype(np.float32)
    
    # Generate patch positions
    z_pos = list(range(0, max(1, D-pd+1), sd))
    if D > pd and (D-pd) not in z_pos: z_pos.append(D-pd)
    y_pos = list(range(0, max(1, H-ph+1), sh))
    if H > ph and (H-ph) not in y_pos: y_pos.append(H-ph)
    x_pos = list(range(0, max(1, W-pw+1), sw))
    if W > pw and (W-pw) not in x_pos: x_pos.append(W-pw)
    
    # Process patches
    for z in tqdm(z_pos, desc="Inference", leave=False):
        for y in y_pos:
            for x in x_pos:
                patch = volume[z:z+pd, y:y+ph, x:x+pw].astype(np.float32)
                actual_shape = patch.shape
                
                # Pad if at volume edges
                if patch.shape != (pd, ph, pw):
                    pad = [(0, max(0, pd-patch.shape[0])), 
                           (0, max(0, ph-patch.shape[1])), 
                           (0, max(0, pw-patch.shape[2]))]
                    patch = np.pad(patch, pad, mode='reflect')
                
                # Normalize
                patch = normalize_volume(patch)
                inp = torch.from_numpy(patch[None, None]).to(device)
                
                # Inference
                with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                    out = torch.sigmoid(model(inp)).squeeze().cpu().numpy()
                
                # Crop to actual size and accumulate
                out = out[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                weight = gauss[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                pred_sum[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += out * weight
                count[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += weight
    
    return pred_sum / np.maximum(count, 1e-8)

print("Inference functions ready")

In [ ]:
# =============================================================================
# MAIN INFERENCE PIPELINE
# =============================================================================

def main():
    # Load test metadata
    test_df = pd.read_csv(cfg.TEST_CSV)
    print(f"Found {len(test_df)} test volumes")
    
    # Initialize model
    print("\nInitializing V4 model...")
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION
    )
    
    # Load weights
    print(f"Loading weights from: {cfg.CHECKPOINT_PATH}")
    if not cfg.CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"Checkpoint not found: {cfg.CHECKPOINT_PATH}")
    
    checkpoint = torch.load(cfg.CHECKPOINT_PATH, map_location=cfg.DEVICE, weights_only=False)
    
    # Handle checkpoint format
    state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
    
    # Print checkpoint info
    if 'epoch' in checkpoint:
        print(f"  Checkpoint epoch: {checkpoint['epoch']}")
    if 'best_score' in checkpoint:
        print(f"  Best Val Dice: {checkpoint['best_score']:.4f}")
    
    # Clean keys (remove module. and _orig_mod. prefixes)
    cleaned_state = {}
    for k, v in state_dict.items():
        key = k.replace('module.', '').replace('_orig_mod.', '')
        cleaned_state[key] = v
    
    # Load weights
    missing, unexpected = model.load_state_dict(cleaned_state, strict=False)
    if missing:
        print(f"  Missing keys: {len(missing)}")
    if unexpected:
        print(f"  Unexpected keys: {len(unexpected)}")
    
    model = model.to(cfg.DEVICE).eval()
    print(f"Model loaded ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")
    
    # Print configuration
    print("\n" + "="*70)
    print("V4 INFERENCE")
    print("="*70)
    print(f"Patch size: {cfg.PATCH_SIZE}")
    print(f"Overlap: {cfg.OVERLAP}")
    if cfg.USE_SIMPLE_THRESHOLD:
        print(f"Post-processing: Simple threshold ({cfg.SIMPLE_THRESHOLD})")
    elif cfg.USE_HYSTERESIS:
        print(f"Post-processing: Hysteresis (H={cfg.THRESHOLD_HIGH}, L={cfg.THRESHOLD_LOW})")
    print(f"Closing: {cfg.USE_CLOSING} (radius={cfg.CLOSING_RADIUS})")
    print(f"Min component: {cfg.MIN_COMPONENT_SIZE}")
    print("="*70)
    
    # Create submission
    with zipfile.ZipFile(cfg.SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for idx, row in test_df.iterrows():
            image_id = row["id"]
            tif_path = cfg.TEST_IMAGES_DIR / f"{image_id}.tif"
            
            print(f"\n[{idx+1}/{len(test_df)}] Processing {image_id}...")
            volume = tifffile.imread(str(tif_path))
            print(f"  Volume shape: {volume.shape}")
            
            # Run inference
            prob_map = sliding_window_inference(
                model=model,
                volume=volume,
                patch_size=cfg.PATCH_SIZE,
                overlap=cfg.OVERLAP,
                device=cfg.DEVICE,
                use_amp=cfg.USE_AMP
            )
            
            # Apply post-processing
            prediction = postprocess(prob_map, cfg)
            
            # Save
            out_path = cfg.OUTPUT_DIR / f"{image_id}.tif"
            tifffile.imwrite(out_path, prediction.astype(np.uint8))
            zf.write(out_path, arcname=f"{image_id}.tif")
            out_path.unlink()
            
            # Cleanup
            del volume, prob_map, prediction
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    print("\n" + "="*70)
    print(f"Submission created: {cfg.SUBMISSION_ZIP}")
    print("="*70)


if __name__ == "__main__":
    main()

## Threshold Tuning Guide for V4

### V4 Model Characteristics
- **Best Val Dice**: 0.6027 (much better than V3's 0.62)
- **Training**: Dice + CE + Skeleton loss with proper warmup
- **Expected probability distribution**: Better calibrated than V3

### Recommended Settings

| Scenario | Settings |
|----------|----------|
| **Default** | `USE_SIMPLE_THRESHOLD=True`, `SIMPLE_THRESHOLD=0.5` |
| Low FG% | Lower `SIMPLE_THRESHOLD` to 0.4 or 0.35 |
| High FG% | Raise `SIMPLE_THRESHOLD` to 0.55 or 0.6 |
| Fragmented vessels | Enable `USE_CLOSING=True`, `CLOSING_RADIUS=1` |
| Too much noise | Increase `MIN_COMPONENT_SIZE` to 100 or 200 |

### If Simple Threshold Doesn't Work Well

Try hysteresis thresholding:
```python
cfg.USE_SIMPLE_THRESHOLD = False
cfg.USE_HYSTERESIS = True
cfg.THRESHOLD_HIGH = 0.6  # Seeds
cfg.THRESHOLD_LOW = 0.3   # Expansion boundary
```

### Target FG%

Based on V2 baseline (LB 0.548): **~10% FG** is a good target.
- If you get < 5% FG: Lower thresholds
- If you get > 15% FG: Raise thresholds